# Análise do dataset

In [55]:
import pandas as pd

df = pd.read_csv("./healthcare_patient_journey.csv")

display(df.head())

,patient_id,age,gender,chronic_condition,admission_type,department,wait_time_min,length_of_stay_days,procedures_count,medication_count,complications,discharge_status,readmitted_30d,total_cost_€,satisfaction_score
0,1,69,male,0,scheduled,Neurology,41,2,0,3,1,referred,1,1440,2
1,2,38,male,0,emergency,Oncology,17,3,1,2,0,recovered,0,2060,3
2,3,81,male,0,scheduled,Neurology,40,2,3,2,0,recovered,0,2110,3
3,4,67,female,1,emergency,ER,7,4,5,9,0,recovered,0,4070,3
4,5,88,male,1,emergency,Cardiology,34,3,7,5,0,recovered,1,3800,3


# Tratamento dos Dados

In [56]:
# Definindo função para colocar título
def titulo(txt):
    print("\n" + "-"*60)
    print(txt)
    print("-"*60)    

In [57]:
# Investigando se há alguma patologia nos dados

# Verificando campos que tem valores nulos
titulo("VERIFICAÇÃO DE VALORES NULOS")
print(df.isnull().any())


# Verificando se há IDs duplicados na base (que deverião ser únicos)
titulo("VERIFICAÇÃO DE IDS DUPLICADOS")
ids_duplicados = df['patient_id'].duplicated()
print(f"Quantidade de IDs duplicados: {ids_duplicados.sum()}")


# Verificando se há valores não numéricos em campos numéricos
titulo("VERIFICAÇÃO DE VALORES NUMÉRICOS")
colunas_numericas = ["age", "chronic_condition", "wait_time_min", "length_of_stay_days", "procedures_count", "medication_count", "complications", "readmitted_30d", "total_cost_€", "satisfaction_score"]

for coluna in colunas_numericas:
    invalidos = pd.to_numeric(df[coluna], errors='coerce').isna()
    print(f"Coluna: {coluna} -> {invalidos.sum()} valores não numéricos")


# Verificando se há valores negativos que podem corromper a base
titulo("VERIFICAÇÃO DE VALORES NEGATIVOS")
for coluna in colunas_numericas:
    negativos = (df[coluna] < 0)
    print(f"Coluna: {coluna} -> {negativos.sum()} valores negativos")


# Verificando se há valores diferentes de 0 ou 1 em campos binários
titulo("VERIFICAÇÃO DE CAMPOS BINÁRIOS")
colunas_binarias = ["chronic_condition", "complications", "readmitted_30d"]

for coluna in colunas_binarias:
    invalidos = (df[coluna] != 0) & (df[coluna] != 1)
    print(f"Coluna: {coluna} -> {invalidos.sum()} valores inválidos")


# Verificando como estão os dados no campo ´total_cost´
titulo("VERIFICAÇÃO DO CAMPO total_cost")
print(df['total_cost_€'])


# Verificando a padronização dos campos categóricos
titulo("VERIFICAÇÃO DE CAMPOS CATEGÓRICOS")

print(f"Gêneros: {df['gender'].unique()}")
print(f"Tipos de admissão: {df['admission_type'].unique()}")
print(f"Departamentos: {df['department'].unique()}")
print(f"Status de alta: {df['discharge_status'].unique()}")


# Verificando a maior e menor idade
titulo("VERIFICAÇÃO DOS LIMITES DAS IDADES")
print(f"Idade mínima: {df['age'].min()}")
print(f"Idade máxima: {df['age'].max()}")



------------------------------------------------------------
VERIFICAÇÃO DE VALORES NULOS
------------------------------------------------------------
patient_id             False
age                    False
gender                 False
chronic_condition      False
admission_type         False
department             False
wait_time_min          False
length_of_stay_days    False
procedures_count       False
medication_count       False
complications          False
discharge_status       False
readmitted_30d         False
total_cost_€           False
satisfaction_score     False
dtype: bool

------------------------------------------------------------
VERIFICAÇÃO DE IDS DUPLICADOS
------------------------------------------------------------
Quantidade de IDs duplicados: 0

------------------------------------------------------------
VERIFICAÇÃO DE VALORES NUMÉRICOS
------------------------------------------------------------
Coluna: age -> 0 valores não numéricos
Coluna: chronic_condi

In [58]:
# Conversão do euro para dólar
dolar = 1.16
df['total_cost_€'] = df['total_cost_€'] * dolar

# Deixando os nomes dos departamentos em minúsculo
df['department'] = df['department'].str.lower()

In [59]:
# Renomeação das colunas
df.rename(columns={
    "patient_id": "id_paciente",
    "age": "idade",
    "gender": "genero",
    "chronic_condition": "condicao_cronica",
    "admission_type": "tipo_admissao",
    "department": "departamento",
    "wait_time_min": "tempo_espera_min",
    "length_of_stay_days": "duracao_estadia_dias",
    "procedures_count": "qtd_procedimentos",
    "medication_count": "qtd_medicamentos",
    "complications": "complicacoes",
    "discharge_status": "status_alta",
    "readmitted_30d": "retorno_apos_30d",
    "total_cost_€": "custo_total_dolares",
    "satisfaction_score": "nivel_satisfacao"
}, inplace=True)

In [60]:
# Tradução dos valores categóricos

df['genero'] = df['genero'].replace({
    'male': 'masculino',
    'female': 'feminino'
})

df['tipo_admissao'] = df['tipo_admissao'].replace({
    'scheduled': 'agendada',
    'emergency': 'emergência'
})

df['departamento'] = df['departamento'].replace({
    'neurology': 'neurologia',
    'oncology': 'oncologia',
    'er': 'pronto socorro',
    'cardiology': 'cardiologia',
    'polyclinic': 'policlínica'
})

df['status_alta'] = df['status_alta'].replace({
    'referred': 'encaminhado',
    'recovered': 'recuperado'
})

%md
# Resultado da Análise e do Tratamento de Colunas

Ao analisar as 15 colunas já vindas do dataset, percebeu-se que não havia nenhum tipo de patologia nos dados. Todos os campos estavam devidamente tratados e com seus devidos tipos:
- Todas as colunas numéricas tinham apenas dados numéricos
- Colunas binárias (de 0 ou 1) não tinham dados diferentes desses dois números
- Não haviam IDs duplicados
- Nenhum tipo de dado nulo dentre os 3000 dados presentes na base
- Não há falta de padronização dentre as colunas categóricas -> Tudo em minúsculo, com exceção do campo `departament` que foi tratado para todos os dados serem minúsculos
- Idades presentes no campo `age` entre 18 e 89 anos, que são dados coerentes e estáveis dentro da base

## Renomeação de Colunas

As colunas estavam de fácil entendimento, porém estavam em inglês, e como nosso dashboard será feito em português decidimos renomeá-las para facilitar a interpretação dos dados e tornar a visualização mais intuitiva para os usuários finais.

Segue abaixo a renomeação realizada:

- `patient_id` → `id_paciente`
- `age` → `idade`
- `gender` → `genero`
- `chronic_condition` → `condicao_cronica`
- `admission_type` → `tipo_admissao`
- `department` → `departamento`
- `wait_time_min` → `tempo_espera_min`
- `length_of_stay_days` → `duracao_estadia_dias`
- `procedures_count` → `qtd_procedimentos`
- `medication_count` → `qtd_medicamentos`
- `complications` → `complicacoes`
- `discharge_status` → `status_alta`
- `readmitted_30d` → `retorno_apos_30d`
- `total_cost_€` → `custo_total_dolares`
- `satisfaction_score` → `nivel_satisfacao`

## Conversão do Campo `custo_total_dolares`
Ao analisar a base e pensarmos na elaboração do dashboard, decidimos converter do euro para o dólar pelo fato de a moeda oficial do dashboard ser o dólar ($).

Levamos em consideração o valor do dólar no dia **22/05/2026**

1 euro (€) = 1,16 dólars americanos ($)

In [61]:
# Subindo base para o spark
# spark.createDataFrame(df).write.mode("overwrite").saveAsTable("df_healthcare_dataset")

# KPIs
- Tempo Médio de Espera p/Departamento
- Período Médio de Estadia p/Departamento
- Quantidade de Pacientes com Doenças Crônicas
- Quantidade de Procedimentos p/Departamento
- Quantidade de Pacientes p/Departamento
- Preço Médio p/Departamento
- Preço Médio Geral
- Nível de Satisfação p/Departamento
- Nível de Satisfação p/Período de Estadia
- Nível de Satisfação Geral
- Satisfação p/Gênero
- Quantidade de Pacientes p/Tipo de Admissão
- Nível de Satisfação p/Tempo de Espera
- Tempo de Espera p/Tipo de Admissão
- Preço por Satisfação (Scatter)
- Quantidade de Medicamentos x Preço x Departamento
- Quantidade de Medicamentos por Quantidade de Procedimentos (Scatter)

In [62]:
# Calculando KPIs

# TEMPO DE ESPERA
titulo("TEMPO MÉDIO DE ESPERA")
print(df["tempo_espera_min"].mean(),"minutos.")

titulo("DESVIO PADRÃO DO TEMPO MÉDIO DE ESPERA")
print(df["tempo_espera_min"].std(),"minutos.")

titulo("TEMPO MÉDIA DE ESPERA P/DEPARTAMENTO")
print(df.groupby("departamento")["tempo_espera_min"].mean(),"minutos.")

titulo("TEMPO MÉDIA DE ESPERA P/DEPARTAMENTO")
print(df.groupby("tipo_admissao")["tempo_espera_min"].mean(),"minutos.")

# ESTADIA
titulo("PERÍODO MÉDIO DE ESTADIA GERAL")
print(df["duracao_estadia_dias"].mean(), "dias.")

titulo("DESVIO PADRÃO DA ESTADIA")
print(df["duracao_estadia_dias"].std(), "dias.")

titulo("ESTADIA MÉDIA POR DEPARTAMENTO")
print(df.groupby("departamento")["duracao_estadia_dias"].mean(),"dias.")

# PACIENTES
titulo("QUANTIDADE TOTAL DE PACIENTES")
print(df["id_paciente"].count(),"pacientes.")

titulo("QUANTIDADE DE PACIENTES POR DEPARTAMENTO")
print(df.groupby("departamento")["id_paciente"].count(),"pacientes.")

titulo("QUANTIDADE DE PACIENTES POR TIPO DE ADMISSÃO")
print(df.groupby("tipo_admissao").size(),"pacientes.")


# DOENÇAS CRÔNICAS
titulo("PACIENTES COM DOENÇA CRÔNICA")
print(df["condicao_cronica"].sum(),"pacientes.")

# PROCEDIMENTOS
titulo("QUANTIDADE TOTAL DE PROCEDIMENTOS")
print(df["qtd_procedimentos"].sum(), "procedimentos.")

titulo("QUANTIDADE TOTAL DE PROCEDIMENTOS POR DEPARTAMENTO")
print(df.groupby("departamento")["qtd_procedimentos"].sum(), "procedimentos.")


# CUSTOS
titulo("PREÇO MÉDIO GERAL")
print(df["custo_total_dolares"].mean())

titulo("DESVIO PADRÃO DO CUSTO")
print(df["custo_total_dolares"].std())

titulo("PREÇO MÉDIO POR DEPARTAMENTO")
print(df.groupby("departamento")["custo_total_dolares"].mean())

titulo("PREÇO MÉDIO POR NÍVEL DE SATISFAÇÃO")
print(df.groupby("nivel_satisfacao")["custo_total_dolares"].mean())


# SATISFAÇÃO
titulo("SATISFAÇÃO MÉDIA GERAL")
print(df["nivel_satisfacao"].mean())

titulo("DESVIO PADRÃO DA SATISFAÇÃO")
print(df["nivel_satisfacao"].std())

titulo("SATISFAÇÃO POR DEPARTAMENTO")
print(df.groupby("departamento")["nivel_satisfacao"].mean())

titulo("SATISFAÇÃO POR GÊNERO")
print(df.groupby("genero")["nivel_satisfacao"].mean())

df["faixa_espera"] = pd.cut(
    df["tempo_espera_min"],
    bins=[0, 15, 30, 60, 120],
    labels=["0-15", "16-30", "31-60", "61-120"]
)
titulo("SATISFAÇÃO POR FAIXA DE TEMPO DE ESPERA")
print(df.groupby("faixa_espera")["nivel_satisfacao"].mean())

df["faixa_estadia"] = pd.cut(
    df["duracao_estadia_dias"],
    bins=[0, 3, 7, 15],
    labels=["0-3", "4-7", "8-15"]
)
titulo("SATISFAÇÃO POR FAIXA DE ESTADIA")
print(df.groupby("faixa_estadia")["nivel_satisfacao"].mean())



------------------------------------------------------------
TEMPO MÉDIO DE ESPERA
------------------------------------------------------------
33.32633333333333 minutos.

------------------------------------------------------------
DESVIO PADRÃO DO TEMPO MÉDIO DE ESPERA
------------------------------------------------------------
17.111558716341126 minutos.

------------------------------------------------------------
TEMPO MÉDIA DE ESPERA P/DEPARTAMENTO
------------------------------------------------------------
departamento
cardiologia       32.814626
neurologia        32.994992
oncologia         33.367893
policlínica       33.766721
pronto socorro    33.666113
Name: tempo_espera_min, dtype: float64 minutos.

------------------------------------------------------------
TEMPO MÉDIA DE ESPERA P/DEPARTAMENTO
------------------------------------------------------------
tipo_admissao
agendada      44.335737
emergência    25.484018
Name: tempo_espera_min, dtype: float64 minutos.

------